# Regex (wyrażenia regularne) od podstaw: moduł `re`, pandas `.str`, Polars i gotowe szablony

Notebook uzupełnia Twój plik `text_operations_str.ipynb` (tam regex pojawia się „przy okazji" w `contains`, `extract`, `replace`; tutaj dostajesz **samą składnię i sposób myślenia**). Prowadzi od zera: *czym jest wzorzec*, jak go czytać, jak zbudować, jak przetestować, aż po lookaround, wydajność i pułapki. Na końcu są **szablony do kopiowania**, ćwiczenia z rozwiązaniami i ściągawka.

**Jak się uczyć regexów (skrót):**
1. Regex to **opis kształtu tekstu**, a nie „magia". Czytaj go od lewej do prawej, element po elemencie.
2. Zawsze testuj na **przykładach, które mają pasować** i **które nie mają pasować**.
3. Buduj wzorzec **iteracyjnie**: najpierw prosty, potem dokładaj ograniczenia.

**Oznaczenia:** ✅ = parametr wymagany, ⬜ = opcjonalny. Kod jest „inline" (bez funkcji); do podglądu dopasowań używamy jednej linii `re.sub(wzorzec, r"[[\g<0>]]", tekst)`, która otacza każde dopasowanie znacznikami `[[ ]]`.

**Spis treści**

0. Setup i dane
1. Intuicja: czym jest regex, jak go czytać, co to `r"..."`
2. Składnia od podstaw: literały, kropka, klasy znaków, kwantyfikatory, kotwice, alternatywa, flagi
3. Zachłanność (greedy) vs leniwość (lazy)
4. Grupy: przechwytujące, nieprzechwytujące, nazwane, odwołania wsteczne
5. Moduł `re`: `search`, `match`, `fullmatch`, `findall`, `finditer`, `sub`, `split`, `compile`
6. Lookaround (zaawansowane): lookahead i lookbehind
7. Regex w pandas (`.str`)
8. Regex w Polars (inny silnik, ważne różnice)
9. **Szablony wzorców** (z testami) i przepisy do czyszczenia danych
10. Jak budować wzorzec krok po kroku (`re.VERBOSE`)
11. Pułapki i wydajność
12. Debugowanie wzorców
13. Ćwiczenia z rozwiązaniami
14. Ściągawki

---

## 0. Setup i dane

In [1]:
import re
import pandas as pd
import polars as pl

# df: te same dane co w Twoim pliku text_operations_str.ipynb
df = pd.DataFrame({
    "product_code": ["PRD-2026-00123", "PRD-2026-00456", "SVC-2025-00789", "PRD-2026-00234"],
    "raw_name": ["  Router TP-Link AC1200 ", "SWITCH netgear 8-port", "cable HDMI 2m", "  Access Point UBIQUITI  "],
    "email": ["jan.kowalski@firma.pl", "anna_nowak@firma.pl", "piotr.wisniewski@firma.pl", "ewa-kowal@firma.pl"],
    "tags": ["electronics,networking,sale", "electronics,networking", "cables,accessories", "electronics,wifi,new"],
    "file_path": ["data/region_north/report_2026.csv", "data/region_south/report_2026.csv",
                  "data/region_east/summary.csv", "data/region_west/report_2026_v2.csv"],
})

# df2: "brudne" kolumny tekstowe, na których ćwiczymy regexy
df2 = pd.DataFrame({
    "phone": ["+48 601 234 567", "601-234-568", "(22) 123 45 67", "tel. 0048 601234569", "brak"],
    "postal": ["00-950 Warszawa", "30-001 Kraków", "80-1 Gdańsk", "50-123 Wrocław", "Poznań 60-001"],
    "date_text": ["2026-03-15", "15.03.2026", "15/03/2026", "5.3.2026", "brak daty"],
    "amount_text": ["1 234,56 zł", "12,50 PLN", "1 234 567,00 zł", "$1,234.56", "brak ceny"],
    "comment": ["Świetny produkt!! polecam #router #wifi", "Kontakt: anna@firma.pl lub +48 600 100 200",
                "to to jest powtórzenie w tekście", "Cena 199 PLN, rabat 15%, #promocja", "Brak uwag"],
    "log": ["2026-03-15 10:21:05 ERROR Brak połączenia z bazą (code=503)", "2026-03-15 10:21:09 INFO Połączono (code=200)",
            "2026-03-15 10:22:41 WARN Wolna odpowiedź (code=429)", "2026-03-15 10:25:00 ERROR Timeout (code=504)", "linia bez formatu"],
})
df2

,phone,postal,date_text,amount_text,comment,log
0,+48 601 234 567,00-950 Warszawa,2026-03-15,"1 234,56 zł",Świetny produkt!! polecam #router #wifi,2026-03-15 10:21:05 ERROR Brak połączenia z ba...
1,601-234-568,30-001 Kraków,15.03.2026,"12,50 PLN",Kontakt: anna@firma.pl lub +48 600 100 200,2026-03-15 10:21:09 INFO Połączono (code=200)
2,(22) 123 45 67,80-1 Gdańsk,15/03/2026,"1 234 567,00 zł",to to jest powtórzenie w tekście,2026-03-15 10:22:41 WARN Wolna odpowiedź (code...
3,tel. 0048 601234569,50-123 Wrocław,5.3.2026,"$1,234.56","Cena 199 PLN, rabat 15%, #promocja",2026-03-15 10:25:00 ERROR Timeout (code=504)
4,brak,Poznań 60-001,brak daty,brak ceny,Brak uwag,linia bez formatu


---

## 1. Intuicja: czym jest regex

**Wyrażenie regularne (regex)** to krótki zapis opisujący **wzorzec tekstu**: np. „dwie cyfry, myślnik, trzy cyfry" (kod pocztowy). Zamiast listy wszystkich możliwych tekstów opisujesz ich **kształt**.

| Po ludzku | Regex | Pasuje do |
|---|---|---|
| „dwie cyfry, myślnik, trzy cyfry" | `\d{2}-\d{3}` | `00-950`, `30-001` |
| „trzy wielkie litery, myślnik, cztery cyfry…" | `[A-Z]{3}-\d{4}` | `PRD-2026`, `SVC-2025` |
| „dowolne znaki, małpa, dowolne znaki, kropka, 2–3 litery" | `.+@.+\.[a-z]{2,3}` | `jan@firma.pl` |

### 1.1. Jak czytać wzorzec

Czytaj od lewej do prawej, **każdy element opisuje jedno miejsce w tekście** (lub kilka, jeśli ma kwantyfikator):

```text
\d{2}-\d{3}
│   │ │   └─ dokładnie 3 razy
│   │ │ └─ cyfra (\d)
│   │ └─ myślnik (zwykły znak, pasuje do samego siebie)
│   └─ dokładnie 2 razy
└─ cyfra (\d)
```

Wzorzec składa się z trzech rodzajów elementów:
1. **Znaki zwykłe** (`a`, `-`, `@`): pasują do samych siebie.
2. **Klasy znaków** (`\d`, `[A-Z]`, `.`): pasują do *jednego znaku z zestawu*.
3. **Modyfikatory** (`+`, `*`, `{2}`, `^`, `$`, `|`, `( )`): mówią *ile razy*, *gdzie*, *albo*, *razem*.

### 1.2. Dlaczego `r"..."` (surowy napis)

W Pythonie `\` w zwykłym napisie jest znakiem specjalnym (`"\n"` to nowa linia). W regex `\d` musi dotrzeć do silnika jako dwa znaki `\` i `d`. Surowy napis `r"\d"` tego pilnuje. **Wzorce regex zawsze pisz jako `r"..."`.**

```python
print(len("\n"), len(r"\n"))        # 1 znak (nowa linia) vs 2 znaki (ukośnik + n)
```

### 1.3. Trzy narzędzia i dwa silniki

| Narzędzie | Silnik regex | Lookaround `(?=…)` | Odwołania wsteczne `\1` |
|---|---|---|---|
| moduł `re` (Python) | Python `re` | ✅ | ✅ |
| pandas `.str.*` | Python `re`; w **pandas 3 z PyArrow** część metod (`contains`, `replace`, `count`) używa silnika Arrow (RE2), zob. **7.2** | ✅ (z automatycznym przejściem na `re`) | ✅ (j.w.) |
| Polars `.str.*` | Rust `regex` | ❌ | ❌ |

Podstawy (klasy, kwantyfikatory, grupy, kotwice, flagi) działają **tak samo wszędzie**. Różnice omawiam w sekcjach 7 i 8.

In [2]:
# podgląd dopasowań: każde dopasowanie otoczone [[ ]]  (\g<0> = całe dopasowanie)
TEXT = "Kody: 00-950 oraz 30-001, ale nie 80-1 ani 123-456."
print(re.sub(r"\d{2}-\d{3}", r"[[\g<0>]]", TEXT))

Kody: [[00-950]] oraz [[30-001]], ale nie 80-1 ani 1[[23-456]].


---

## 2. Składnia od podstaw

W każdej podsekcji: tabela elementów i komórka z przykładem. Odpowiednik na pytanie „co tu zostało dopasowane?" pokazuje `[[…]]`.

### 2.1. Znaki zwykłe i kropka

| Element | Znaczenie |
|---|---|
| `a`, `5`, `-`, `@`, spacja | Pasuje do **samego siebie** |
| `.` | **Dowolny jeden znak** (oprócz nowej linii; z flagą `re.S` także nowa linia) |
| `\.` | **Dosłowna kropka** (ukośnik „wyłącza" znaczenie specjalne) |

Znaki specjalne, które trzeba poprzedzić `\`, żeby były dosłowne: `. ^ $ * + ? { } [ ] \ | ( )`. Gotowy tekst „oznaczający sam siebie" wyprodukuje `re.escape(...)`.

In [3]:
txt = "plik.csv plik_csv plikXcsv"
print(re.sub(r"plik.csv", r"[[\g<0>]]", txt))        # kropka = dowolny znak: pasuje do wszystkich trzech
print(re.sub(r"plik\.csv", r"[[\g<0>]]", txt))       # \. = dosłowna kropka: tylko pierwszy
print(re.escape("cena (zł) = 5.00+"))                # re.escape: bezpiecznie zamienia tekst na wzorzec dosłowny

[[plik.csv]] [[plik_csv]] [[plikXcsv]]
[[plik.csv]] plik_csv plikXcsv
cena\ \(zł\)\ =\ 5\.00\+


### 2.2. Klasy znaków: `[...]` i skróty

**Klasa** = „jeden znak z tego zestawu".

| Zapis | Znaczenie | Przykład |
|---|---|---|
| `[abc]` | `a` lub `b` lub `c` | `gr[ae]y` → `gray`, `grey` |
| `[a-z]`, `[A-Z]`, `[0-9]` | zakres | `[A-Za-z0-9]` litery i cyfry ASCII |
| `[^abc]` | **dowolny znak oprócz** `a`, `b`, `c` | `[^0-9]` = nie-cyfra |
| `[a-ząćęłńóśźż]` | zakres z polskimi literami | trzeba wymienić ręcznie |
| `\d` / `\D` | cyfra / nie-cyfra | |
| `\w` / `\W` | „znak słowa": litera (także polska), cyfra lub `_` / odwrotność | |
| `\s` / `\S` | biały znak (spacja, tab, nowa linia, NBSP) / odwrotność | |

Wewnątrz `[...]` większość znaków specjalnych jest zwykła (`[.+*]` = kropka, plus lub gwiazdka), ale `-` (zakres), `^` (na początku), `]` i `\` mają znaczenie.

> **Litery z polskimi znakami:** `\w` w Pythonie (dla tekstu Unicode) dopasowuje też `ą`, `ł`, `ż` itd., ale także cyfry i `_`. Same litery (Unicode): `[^\W\d_]`. **Uwaga:** w pandas 3 (domyślny typ tekstowy `str` z PyArrow) `contains`/`replace`/`count` mogą traktować `\w` jako **tylko ASCII** (szczegóły i obejście w sekcji 7.2).

In [4]:
txt = "Zażółć gęślą jaźń 2026_v2!"
print(re.sub(r"\w+", r"[[\g<0>]]", txt))              # \w: litery (także polskie), cyfry i _
print(re.sub(r"[^\W\d_]+", r"[[\g<0>]]", txt))        # same litery (Unicode)
print(re.sub(r"[a-z]+", r"[[\g<0>]]", txt))           # [a-z] widzi tylko ASCII: polskie litery "rozbijają" wyrazy
print(re.sub(r"[^a-zA-Z ]+", r"[[\g<0>]]", txt))      # negacja: wszystko oprócz liter ASCII i spacji
print(re.findall(r"gr[ae]y", "gray grey groy"))      # [ae]: jedna z dwóch liter

[[Zażółć]] [[gęślą]] [[jaźń]] [[2026_v2]]!
[[Zażółć]] [[gęślą]] [[jaźń]] 2026_[[v]]2!
Z[[a]]żółć [[g]]ęś[[l]]ą [[ja]]źń 2026_[[v]]2!
Za[[żółć]] g[[ęś]]l[[ą]] ja[[źń]] [[2026_]]v[[2!]]
['gray', 'grey']


### 2.3. Kwantyfikatory: ile razy

Kwantyfikator stoi **po** elemencie, którego dotyczy (znaku, klasie albo grupie).

| Zapis | Znaczenie | Przykład | Pasuje do |
|---|---|---|---|
| `?` | 0 lub 1 raz (element opcjonalny) | `colou?r` | `color`, `colour` |
| `*` | 0 lub więcej | `ab*` | `a`, `ab`, `abbb` |
| `+` | 1 lub więcej | `\d+` | `7`, `2026` |
| `{n}` | dokładnie n | `\d{4}` | `2026` |
| `{n,m}` | od n do m | `\d{2,4}` | `12`, `123`, `1234` |
| `{n,}` | n lub więcej | `\d{3,}` | `123`, `12345` |
| `{,m}` | od 0 do m | `\d{,2}` | `""`, `1`, `12` |

In [5]:
txt = "color colour 7 2026 12345 ab abbb"
print(re.findall(r"colou?r", txt))                    # ? opcjonalne u
print(re.findall(r"\d+", txt))                        # + jedna lub więcej cyfr
print(re.findall(r"\b\d{4}\b", txt))                  # dokładnie 4 cyfry jako osobny wyraz
print(re.findall(r"\d{2,4}", "12345678"))             # 2–4 cyfry; "chciwie": bierze maksimum
print(re.findall(r"ab*", txt))                        # a, po którym 0+ liter b

['color', 'colour']
['7', '2026', '12345']
['2026']
['1234', '5678']
['ab', 'abbb']


### 2.4. Kotwice: gdzie w tekście

Kotwice **nie zajmują znaków**, tylko wskazują miejsce.

| Zapis | Znaczenie |
|---|---|
| `^` | początek tekstu (z flagą `re.M` także początek **każdej linii**) |
| `$` | koniec tekstu (z `re.M` koniec każdej linii; **uwaga:** pasuje też przed końcowym `\n`) |
| `\A` / `\Z` | ścisły początek / ścisły koniec całego tekstu (nie zależą od `re.M`) |
| `\b` | granica wyrazu (między `\w` a `\W` lub brzegiem tekstu) |
| `\B` | miejsce, które **nie** jest granicą wyrazu |

In [6]:
txt = "kot kotek kotwica okot"
print(re.findall(r"kot", txt))                        # 4× (w tym wewnątrz wyrazów)
print(re.findall(r"\bkot\b", txt))                    # tylko cały wyraz "kot"
print(re.findall(r"\bkot\w*", txt))                   # wyrazy zaczynające się od "kot"
print(re.findall(r"^\d+", "123abc"), re.findall(r"^\d+", "abc123"), re.findall(r"\d+$", "abc123"))   # ^ i $
multi = "linia 1\nlinia 2\nlinia 3"
print(re.findall(r"^linia \d$", multi), re.findall(r"^linia \d$", multi, flags=re.M))                # re.M: ^ $ działają na liniach

['kot', 'kot', 'kot', 'kot']
['kot']
['kot', 'kotek', 'kotwica']
['123'] [] ['123']
[] ['linia 1', 'linia 2', 'linia 3']


### 2.5. Alternatywa `|` i nawiasy `( )`

`a|b` = „a **albo** b". Alternatywa obejmuje **cały** lewy i prawy fragment; ograniczasz ją nawiasami.

| Zapis | Znaczenie |
|---|---|
| `kot\|pies` | `kot` albo `pies` |
| `gr(a\|e)y` | `gray` albo `grey` (nawias ogranicza alternatywę) |
| `gra\|ey` | `gra` albo `ey` (**bez** nawiasów: inny sens!) |
| `(ab)+` | `ab`, `abab`, `ababab` (kwantyfikator na całej grupie) |

In [7]:
print(re.findall(r"kot|pies", "kot i pies i ptak"))
print(re.findall(r"gr(?:a|e)y", "gray grey"), re.findall(r"gra|ey", "gray grey"))     # różnica: nawiasy
print(re.sub(r"(ab)+", r"[[\g<0>]]", "ab abab aab"))                                  # kwantyfikator dla całej grupy

['kot', 'pies']
['gray', 'grey'] ['gra', 'ey']
[[ab]] [[abab]] a[[ab]]


### 2.6. Flagi (tryby)

| Flaga | Skrót | Inline | Działanie |
|---|---|---|---|
| `re.IGNORECASE` | `re.I` | `(?i)` | Ignoruj wielkość liter (w Unicode obejmuje polskie litery) |
| `re.MULTILINE` | `re.M` | `(?m)` | `^` i `$` dla każdej linii |
| `re.DOTALL` | `re.S` | `(?s)` | `.` pasuje także do nowej linii |
| `re.VERBOSE` | `re.X` | `(?x)` | Spacje i komentarze `#` we wzorcu są ignorowane (czytelne długie wzorce) |
| `re.ASCII` | `re.A` | `(?a)` | `\w`, `\d`, `\s`, `\b` tylko dla ASCII |

Flagi łączysz operatorem `|`: `re.I | re.M`. W pandas przekazujesz je parametrem `flags=`, w Polars wyłącznie inline (`(?i)`).

In [8]:
print(re.findall(r"router", "Router ROUTER router"), re.findall(r"router", "Router ROUTER router", flags=re.I))
print(re.findall(r"(?i)żółć", "ŻÓŁĆ żółć"))                                     # (?i) inline, także polskie litery
print(re.findall(r"a.b", "a\nb a-b"), re.findall(r"a.b", "a\nb a-b", flags=re.S))        # kropka a nowa linia
print(re.findall(r"\d", "٣4 5"), re.findall(r"\d", "٣4 5", flags=re.A))                    # \d w Unicode łapie też cyfry z innych pism (٣)

['router'] ['Router', 'ROUTER', 'router']
['ŻÓŁĆ', 'żółć']
['a-b'] ['a\nb', 'a-b']
['٣', '4', '5'] ['4', '5']


---

## 3. Zachłanność (greedy) vs leniwość (lazy)

Kwantyfikatory `* + ? {n,m}` są domyślnie **zachłanne**: biorą **jak najwięcej**, a cofają się tylko, gdy reszta wzorca nie pasuje. Dodanie `?` po kwantyfikatorze robi go **leniwym** (bierze jak najmniej).

| Zachłanny | Leniwy | Znaczenie |
|---|---|---|
| `.*` | `.*?` | dowolny tekst, jak najdłuższy / jak najkrótszy |
| `.+` | `.+?` | co najmniej jeden znak |
| `\d{2,5}` | `\d{2,5}?` | zachłannie 5 cyfr / leniwie 2 |

Najczęstszy błąd: wzorzec `".*"` na tekście z kilkoma cudzysłowami łapie wszystko od pierwszego do **ostatniego**.

In [9]:
txt = 'Powiedział "tak" i "nie" oraz "może".'
print(re.findall(r'".*"', txt))               # zachłannie: jeden wielki fragment
print(re.findall(r'".*?"', txt))              # leniwie: każdy cytat osobno
print(re.findall(r'"([^"]*)"', txt))          # najlepsze: "dowolne znaki oprócz cudzysłowu" (szybsze i bez cofania)

html = "<b>pogrubione</b> i <i>kursywa</i>"
print(re.findall(r"<.+>", html), re.findall(r"<.+?>", html))     # tagi: zachłanny vs leniwy

['"tak" i "nie" oraz "może"']
['"tak"', '"nie"', '"może"']
['tak', 'nie', 'może']
['<b>pogrubione</b> i <i>kursywa</i>'] ['<b>', '</b>', '<i>', '</i>']


> **Reguła:** gdy chcesz „do najbliższego znaku X", wolisz `[^X]*` niż `.*?`. Jest czytelniejsze i szybsze, bo silnik nie próbuje cofać się po wielu możliwościach. Python 3.11+ ma też kwantyfikatory zaborcze (`*+`, `++`, `?+`) i grupy atomowe `(?>...)`, które zakazują cofania (wydajność, zob. sekcja 11).

---

## 4. Grupy: przechwytywanie, nazwy, odwołania wsteczne

Nawias `( )` robi dwie rzeczy: **grupuje** fragment wzorca (dla kwantyfikatora lub alternatywy) i **zapamiętuje** dopasowany tekst („przechwytuje") pod numerem 1, 2, 3… liczonym po kolejności **otwierających nawiasów**.

| Składnia | Nazwa | Efekt |
|---|---|---|
| `(...)` | grupa przechwytująca | Zapamiętuje tekst: numer 1, 2, … |
| `(?:...)` | grupa **nieprzechwytująca** | Tylko grupuje (nie zapamiętuje): lżejsza, nie psuje numeracji |
| `(?P<nazwa>...)` | grupa **nazwana** | Zapamiętuje pod nazwą (w pandas nazwa grupy staje się nazwą kolumny) |
| `\1`, `\2` | odwołanie wsteczne | W **tym samym wzorcu** powtarza tekst z grupy 1, 2 |
| `(?P=nazwa)` | odwołanie do nazwanej | To samo przez nazwę |
| `\1` / `\g<1>` / `\g<nazwa>` | w **zamienniku** `re.sub` | Wstawia tekst zapamiętany w grupie |

In [10]:
code_txt = "PRD-2026-00123"
m = re.search(r"([A-Z]{3})-(\d{4})-(\d{5})", code_txt)          # 3 grupy przechwytujące
print(m.group(0), "|", m.group(1), m.group(2), m.group(3), "|", m.groups())

m = re.search(r"(?P<typ>[A-Z]{3})-(?P<rok>\d{4})-(?P<nr>\d{5})", code_txt)   # grupy nazwane
print(m.groupdict(), "| span całego:", m.span(), "| span roku:", m.span("rok"))

# findall z grupami zwraca KROTKI grup (nie całe dopasowania)
print(re.findall(r"(\d{2})-(\d{3})", "00-950 i 30-001"))
print(re.findall(r"\d{2}-\d{3}", "00-950 i 30-001"))              # bez grup: całe dopasowania

# grupa nieprzechwytująca: grupuje dla kwantyfikatora, ale nie wchodzi do wyniku
print(re.findall(r"(?:\d{3}[ -]?){2}\d{3}", "601 234 567"), re.findall(r"(\d{3}[ -]?){2}\d{3}", "601 234 567"))

# odwołanie wsteczne w tym samym wzorcu: powtórzone słowo
print(re.search(r"\b(\w+)\s+\1\b", "to to jest powtórzenie").group(0))
print(re.search(r"\b(?P<w>\w+)\s+(?P=w)\b", "bardzo bardzo dobrze").group("w"))

# zamiennik z odwołaniem: "Nazwisko Imię" → "Imię Nazwisko"
print(re.sub(r"(\w+) (\w+)", r"\2 \1", "Kowalski Jan"), re.sub(r"(?P<n>\w+) (?P<i>\w+)", r"\g<i> \g<n>", "Kowalski Jan"))

PRD-2026-00123 | PRD 2026 00123 | ('PRD', '2026', '00123')
{'typ': 'PRD', 'rok': '2026', 'nr': '00123'} | span całego: (0, 14) | span roku: (4, 8)
[('00', '950'), ('30', '001')]
['00-950', '30-001']
['601 234 567'] ['234 ']
to to
bardzo
Jan Kowalski Jan Kowalski


### 4.1. Obiekt dopasowania (`Match`)

`re.search`, `re.match`, `re.fullmatch` zwracają obiekt `Match` albo `None` (brak dopasowania; **zawsze sprawdź `if m:`** przed użyciem).

| Metoda | Zwraca |
|---|---|
| `m.group()` / `m.group(0)` | Całe dopasowanie |
| `m.group(1)`, `m.group("nazwa")` | Tekst grupy (`None`, jeśli grupa nie uczestniczyła) |
| `m.groups()` | Krotka wszystkich grup (od 1) |
| `m.groupdict()` | Słownik nazwanych grup |
| `m.start()`, `m.end()`, `m.span()` | Pozycje w tekście (od 0, koniec wyłączony) |
| `m.lastindex`, `m.re`, `m.string` | Ostatnia grupa, wzorzec, tekst wejściowy |

---

## 5. Moduł `re`: funkcje

| Funkcja | Parametry (✅ / ⬜) | Zwraca | Użycie |
|---|---|---|---|
| `re.search(pattern, string, flags)` | ✅ `pattern`, ✅ `string`; ⬜ `flags` | `Match` / `None` | **Pierwsze** dopasowanie gdziekolwiek w tekście |
| `re.match(pattern, string, flags)` | j.w. | `Match` / `None` | Dopasowanie **od początku** tekstu (nie musi do końca) |
| `re.fullmatch(pattern, string, flags)` | j.w. | `Match` / `None` | Dopasowanie **całego** tekstu (walidacja formatu) |
| `re.findall(pattern, string, flags)` | j.w. | lista (teksty lub krotki grup) | **Wszystkie** dopasowania |
| `re.finditer(pattern, string, flags)` | j.w. | iterator `Match` | Wszystkie dopasowania **z pozycjami i grupami** |
| `re.sub(pattern, repl, string, count, flags)` | ✅ `pattern`, ✅ `repl`, ✅ `string`; ⬜ `count` (0 = wszystkie), `flags` | tekst | Zamiana; `repl` to tekst z `\1`/`\g<n>` **lub funkcja** |
| `re.subn(...)` | j.w. | `(tekst, liczba zamian)` | Jak `sub`, plus liczba zamian |
| `re.split(pattern, string, maxsplit, flags)` | ✅ `pattern`, ✅ `string`; ⬜ `maxsplit`, `flags` | lista | Dzielenie po wzorcu (grupy przechwytujące **trafiają do wyniku**) |
| `re.compile(pattern, flags)` | ✅ `pattern`; ⬜ `flags` | obiekt wzorca | Kompilacja do wielokrotnego użycia (`p.search(...)`, `p.sub(...)`) |
| `re.escape(text)` | ✅ `text` | tekst | Zamienia tekst na wzorzec dosłowny |

**`search` vs `match` vs `fullmatch`** (zob. też sekcja 13 Twojego pliku, gdzie to samo robią `contains`, `match`, `fullmatch` w pandas):

| Funkcja | `re.xxx(r"\d{3}", "123abc")` | `re.xxx(r"\d{3}", "abc123")` | `re.xxx(r"\d{3}", "123")` |
|---|---|---|---|
| `search` | dopasowanie | dopasowanie | dopasowanie |
| `match` | dopasowanie | `None` (nie od początku) | dopasowanie |
| `fullmatch` | `None` (reszta tekstu) | `None` | dopasowanie |

In [11]:
for func in (re.search, re.match, re.fullmatch):
    print(f"{func.__name__:10s}", [bool(func(r"\d{3}", t)) for t in ("123abc", "abc123", "123")])

# finditer: pozycje i grupy każdego dopasowania
for m in re.finditer(r"(?P<h>\d{2}):(?P<mi>\d{2})", "start 08:15, koniec 17:45"):
    print(m.group(), m.span(), m.groupdict())

search     [True, True, True]
match      [True, False, True]
fullmatch  [False, False, True]
08:15 (6, 11) {'h': '08', 'mi': '15'}
17:45 (20, 25) {'h': '17', 'mi': '45'}


In [12]:
# sub: zamiana tekstem, z odwołaniem do grup i z funkcją (callback)
print(re.sub(r"\s+", " ", "dużo    spacji\ti\nznaków"))                              # kilka białych znaków → jedna spacja
print(re.sub(r"(\d{2})\.(\d{2})\.(\d{4})", r"\3-\2-\1", "dzisiaj 15.03.2026"))        # DD.MM.RRRR → RRRR-MM-DD
print(re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "3 jabłka i 10 gruszek"))    # funkcja jako zamiennik: liczby ×2
print(re.subn(r"a", "A", "banana"), re.sub(r"a", "A", "banana", count=2))           # subn: liczba zamian, count: limit

# split: dzielenie po wzorcu (separatory różnej postaci); grupa w nawiasach ZOSTAJE w wyniku
print(re.split(r"[,;|]\s*", "a, b;c |d"))
print(re.split(r"([,;])", "a,b;c"))                                                   # z grupą: separatory też w liście
print(re.split(r"\s+", "  ala ma   kota ".strip(), maxsplit=1))

# compile: wzorzec raz, użycie wielokrotnie (czytelność i szybkość w pętlach)
pat = re.compile(r"(?P<typ>[A-Z]{3})-(?P<rok>\d{4})-(?P<nr>\d{5})")
print(pat.groups, pat.groupindex, [pat.fullmatch(c) is not None for c in df["product_code"]])

dużo spacji i znaków
dzisiaj 2026-03-15
6 jabłka i 20 gruszek
('bAnAnA', 3) bAnAna
['a', 'b', 'c ', 'd']
['a', ',', 'b', ';', 'c']
['ala', 'ma   kota']
3 {'typ': 1, 'rok': 2, 'nr': 3} [True, True, True, True]


---

## 6. Lookaround (zaawansowane): „zerkanie" bez zjadania znaków

**Lookaround** sprawdza, **co jest przed lub za** bieżącym miejscem, ale **nie wchodzi do dopasowania** (nie zjada znaków). Dzięki temu możesz powiedzieć: „liczba, **po której** następuje PLN" i dostać samą liczbę.

| Składnia | Nazwa | Znaczenie |
|---|---|---|
| `X(?=Y)` | lookahead pozytywny | `X`, **po którym następuje** `Y` |
| `X(?!Y)` | lookahead negatywny | `X`, **po którym NIE następuje** `Y` |
| `(?<=Y)X` | lookbehind pozytywny | `X`, **przed którym jest** `Y` |
| `(?<!Y)X` | lookbehind negatywny | `X`, **przed którym NIE ma** `Y` |

Ograniczenia: w module `re` (i pandas) lookbehind wymaga wzorca o **stałej długości** (`(?<=Cena )` OK, `(?<=\s+)` błąd). **Polars (silnik Rust) w ogóle nie obsługuje lookaround ani odwołań wstecznych** (zob. sekcja 8).

In [13]:
txt = "Cena 199 PLN, rabat 15%, 20 sztuk, 35 USD, nr 7"
print(re.findall(r"\d+(?= PLN)", txt))              # liczba, po której jest " PLN" (PLN nie wchodzi do wyniku)
print(re.findall(r"\d+(?![\d%])", txt))             # liczby, po których NIE ma % (całej liczby, nie jej części)
print(re.findall(r"(?<=Cena )\d+", txt))            # liczba po "Cena "
print(re.findall(r"(?<!nr )\b\d+\b", txt))          # liczby bez poprzedzającego "nr "
print(re.findall(r"(?=(\d\d))", "12345"))           # nakładające się dopasowania (zwykłe findall dałoby 12, 34)

try:
    re.compile(r"(?<=\s+)\d")                       # lookbehind o zmiennej długości
except re.error as e:
    print("błąd:", e)

['199']
['199', '20', '35', '7']
['199']
['199', '15', '20', '35']
['12', '23', '34', '45']
błąd: look-behind requires fixed-width pattern


In [14]:
# praktyczne zastosowania lookaround
print(re.sub(r"\B(?=(\d{3})+(?!\d))", " ", "1234567"))                      # separatory tysięcy: 1 234 567
print(re.split(r"(?<=[a-z])(?=[A-Z])", "camelCaseTextHere"))               # cięcie CamelCase bez zjadania liter
print(re.sub(r"(?<!^)(?=[A-Z])", "_", "CamelCaseText").lower())            # CamelCase → snake_case

# walidacja hasła: kilka warunków naraz (każdy lookahead sprawdza inny warunek od początku tekstu)
pw = r"(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}"
print([bool(re.fullmatch(pw, p)) for p in ["abc", "Abcdefg1", "abcdefgh1", "ABCDEFGH1a"]])

1 234 567
['camel', 'Case', 'Text', 'Here']
camel_case_text
[False, True, False, True]


---

## 7. Regex w pandas (`.str`)

Metody `Series.str.*` używają silnika Pythona `re` (pełna składnia, w tym lookaround i odwołania wsteczne). Wynik jest **wektorowy**: wzorzec stosowany do każdego elementu kolumny, `NaN` zostaje `NaN`.

| Metoda | Parametry (✅ / ⬜) | Zwraca | Użycie |
|---|---|---|---|
| `contains(pat, case, flags, na, regex)` | ✅ `pat`; ⬜ `case=True`, `flags=0`, `na=None`, `regex=True` | bool Series | Czy wzorzec **występuje gdziekolwiek** (filtrowanie) |
| `match(pat, case, flags, na)` | ✅ `pat`; ⬜ reszta | bool Series | Dopasowanie **od początku** tekstu |
| `fullmatch(pat, case, flags, na)` | ✅ `pat`; ⬜ reszta | bool Series | Dopasowanie **całego** tekstu (walidacja) |
| `extract(pat, flags, expand)` | ✅ `pat` (**co najmniej 1 grupa**); ⬜ `flags=0`, `expand=True` | DataFrame (kolumna na grupę; nazwa = nazwa grupy) lub Series | **Pierwsze** dopasowanie: grupy do kolumn |
| `extractall(pat, flags)` | ✅ `pat` (z grupami); ⬜ `flags` | DataFrame z MultiIndex `(wiersz, match)` | **Wszystkie** dopasowania |
| `findall(pat, flags)` | ✅ `pat`; ⬜ `flags` | Series list | Lista wszystkich dopasowań w komórce |
| `count(pat, flags)` | ✅ `pat`; ⬜ `flags` | int Series | Liczba dopasowań |
| `replace(pat, repl, n, case, flags, regex)` | ✅ `pat`, ✅ `repl`; ⬜ `n=-1`, `case`, `flags`, **`regex=False`** | Series | Zamiana; `repl` może zawierać `\1` lub być **funkcją** (wymaga `regex=True`) |
| `split(pat, n, expand, regex)` | ⬜ wszystkie; `pat` o długości > 1 jest regexem | Series list / DataFrame | Dzielenie po wzorcu |
| `startswith`, `endswith` | ✅ `pat` (tekst lub krotka) | bool | **Nie** regex: zwykły prefiks/sufiks |

> **Uwaga (pandas ≥ 2.0):** `str.replace` ma domyślnie **`regex=False`** (dosłowny tekst). Aby użyć wzorca, **zawsze ustaw `regex=True`**. W `contains`, `match`, `extract` wzorzec jest regexem domyślnie. W `contains` używaj `na=False`, gdy budujesz maskę do filtrowania (inaczej `NaN` psuje indeksowanie logiczne).

In [15]:
# a) filtrowanie: contains / match / fullmatch na kodach produktów
print(df["product_code"].str.contains(r"2026", na=False).tolist())                 # 2026 gdziekolwiek
print(df["product_code"].str.match(r"PRD").tolist())                               # zaczyna się od PRD
print(df["product_code"].str.fullmatch(r"[A-Z]{3}-\d{4}-\d{5}").tolist())          # cały tekst ma ten format
print(df[df["product_code"].str.contains(r"^PRD-2026-", na=False)]["product_code"].tolist())   # maska do filtrowania
print(df["raw_name"].str.contains("router|switch", case=False, na=False).tolist()) # alternatywa + ignorowanie wielkości liter

# b) extract z grupami nazwanymi: kod produktu rozbity na kolumny
parts = df["product_code"].str.extract(r"(?P<typ>[A-Z]{3})-(?P<rok>\d{4})-(?P<nr>\d{5})")
parts["rok"] = parts["rok"].astype(int)
parts

[True, True, False, True]
[True, True, False, True]
[True, True, True, True]
['PRD-2026-00123', 'PRD-2026-00456', 'PRD-2026-00234']
[True, True, False, False]


,typ,rok,nr
0,PRD,2026,00123
1,PRD,2026,00456
2,SVC,2025,00789
3,PRD,2026,00234


In [16]:
# c) e-mail: walidacja formatu, część lokalna i domena; wykrycie separatora w części lokalnej
print(df["email"].str.fullmatch(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+").tolist())
print(df["email"].str.extract(r"^(?P<lokalna>[^@]+)@(?P<domena>[\w.-]+)$"))
print(df["email"].str.extract(r"^[^@]*?([._-])[^@]*@", expand=False).tolist())    # jaki separator (. _ -) występuje w nazwie

# d) tagi: ile, które słowa, całe wyrazy (\b)
print(df["tags"].str.count(",").add(1).tolist(), df["tags"].str.contains(r"\bsale\b").tolist(), df["tags"].str.findall(r"[a-z]+").tolist())

# e) ścieżki plików: region, nazwa, rozszerzenie, opcjonalna wersja
print(df["file_path"].str.extract(r"region_(?P<region>\w+)/"))
print(df["file_path"].str.extract(r"(?P<plik>[^/]+?)(?:_v(?P<wersja>\d+))?\.(?P<ext>\w+)$"))   # wersja opcjonalna → NaN gdy brak

[True, True, True, True]
            lokalna    domena
0      jan.kowalski  firma.pl
1        anna_nowak  firma.pl
2  piotr.wisniewski  firma.pl
3         ewa-kowal  firma.pl
['.', '_', '.', '-']
[3, 2, 2, 3] [True, False, False, False] [['electronics', 'networking', 'sale'], ['electronics', 'networking'], ['cables', 'accessories'], ['electronics', 'wifi', 'new']]
  region
0  north
1  south
2   east
3   west
          plik wersja  ext
0  report_2026    NaN  csv
1  report_2026    NaN  csv
2      summary    NaN  csv
3  report_2026      2  csv


In [17]:
# f) replace: zawsze regex=True; odwołania \1; funkcja jako zamiennik (dostaje obiekt Match)
print(df2["date_text"].str.replace(r"(\d{1,2})\.(\d{1,2})\.(\d{4})", r"\3-\2-\1", regex=True).tolist())    # DD.MM.RRRR → RRRR-MM-DD
print(df2["date_text"].str.replace(r"^(?P<d>\d{1,2})\.(?P<m>\d{1,2})\.(?P<y>\d{4})$",
                                   lambda m: f"{m['y']}-{int(m['m']):02d}-{int(m['d']):02d}", regex=True).tolist())   # z dopełnieniem zerami
print(pd.Series(["abc"]).str.replace(".", "-"), pd.Series(["abc"]).str.replace(".", "-", regex=True), sep="\n")   # regex=False vs True

# g) extractall: wszystkie hashtagi (MultiIndex: wiersz, numer dopasowania) i prostsze findall
print(df2["comment"].str.extractall(r"(?P<tag>#\w+)"))
print(df2["comment"].str.findall(r"#\w+").tolist())

# h) NaN: maski i na=False
s = pd.Series(["a1", None, "b2"])
print(s.str.contains(r"\d").tolist(), s.str.contains(r"\d", na=False).tolist())

['2026-03-15', '2026-03-15', '15/03/2026', '2026-3-5', 'brak daty']
['2026-03-15', '2026-03-15', '15/03/2026', '2026-03-05', 'brak daty']
0    abc
dtype: str
0    ---
dtype: str
               tag
  match           
0 0        #router
  1          #wifi
3 0      #promocja
[['#router', '#wifi'], [], [], ['#promocja'], []]
[True, False, True] [True, False, True]


### 7.1. Najczęstsze pułapki w pandas

| Problem | Przyczyna / rozwiązanie |
|---|---|
| `str.replace(".", "x")` nic nie robi tak, jak oczekujesz | `regex=False` jest domyślne: kropka jest dosłowna. Dodaj `regex=True` (i wtedy `\.` dla dosłownej kropki) |
| `UserWarning: This pattern has match groups` w `contains` | Użyj grup nieprzechwytujących `(?:…)` albo `extract` |
| `extract` zwraca `NaN` | Brak dopasowania albo brak grup w wzorcu (wzorzec musi mieć `( )`) |
| `contains` zwraca `NaN` w masce i filtr się wywala | `na=False` |
| Wzorzec z polskimi literami i `case=False` | Działa (Unicode), ale `[a-z]` nie obejmuje polskich liter: użyj `[^\W\d_]` |
| Wolne `apply(lambda x: re.search(...))` | Preferuj wektorowe `.str.*` (kilkukrotnie szybsze) |
| `\w`, `\b` nie widzą polskich liter w `contains`/`replace`/`count` | pandas 3 + PyArrow: silnik RE2 (ASCII). Dodaj `flags=re.U` lub zob. **7.2** |
| `ArrowInvalid: invalid escape sequence: \u` | RE2 nie zna `\uXXXX`. Użyj `\xhh` (do U+00FF), znaku wprost w zwykłym napisie albo `flags=re.U` |

### 7.2. Dwa silniki regex w jednym pandas (ważne w pandas 3 z PyArrow)

W **pandas 3** domyślny typ tekstowy `str` jest oparty na **PyArrow**. Wtedy część metod (w testach: `contains`, `replace`, `count`) używa silnika **Arrow (RE2)**, a inne (`extract`, `findall`, `split`) silnika Pythona `re`. Konsekwencje dla polskiego tekstu:

| Cecha | Python `re` | Arrow (RE2) |
|---|---|---|
| `\w`, `\d`, `\b`, `[^\W\d_]` | Unicode (widzi `ą`, `ł`, `ż`) | **tylko ASCII** (polskie litery nie są „literami słowa") |
| `\s` | obejmuje NBSP (`\xa0`) | tylko ASCII-owe białe znaki |
| `\uXXXX` | ✅ | ❌ (`ArrowInvalid`); użyj `\xhh` lub znaku wprost |
| Lookaround, `\1` | ✅ | ❌, ale pandas przechodzi wtedy na `re` automatycznie |
| `\p{L}` | ❌ | ✅ |

**Obejścia (od najprostszego):**
1. **`flags=re.U`** (lub inna niezerowa flaga) w `contains`/`replace`/`count`: wymusza silnik Pythona; nie szkodzi w starszych wersjach pandas.
2. **Jawne klasy** zamiast `\w`: `[A-Za-ząćęłńóśźżĄĆĘŁŃÓŚŹŻ]` (identycznie w każdym silniku).
3. Tekst jako `object` / `string[python]`: `s.astype(object)`.
4. Duże dane i spójna semantyka Unicode: Polars (`\w` jest tam domyślnie Unicode).

Komórka poniżej pokazuje, jak to wygląda **u Ciebie** (wynik zależy od wersji pandas i obecności PyArrow; w pandas 2.x wszystkie cztery wiersze dadzą `True`).

In [18]:
tekst = pd.Series(["żółć", "gęślą", "abc"])
print(pd.__version__, "|", tekst.dtype, "|", type(tekst.array).__name__)
print("domyślnie             :", tekst.str.contains(r"^\w+$").tolist())                      # pandas 3 + PyArrow: [False, False, True] (RE2: \w tylko ASCII)
print("flags=re.U (Python re):", tekst.str.contains(r"^\w+$", flags=re.U).tolist())            # wymusza silnik Pythona
print("astype(object)        :", tekst.astype(object).str.contains(r"^\w+$").tolist())        # tekst jako obiekty Pythona
print("jawna klasa liter PL  :", tekst.str.contains(r"^[A-Za-ząćęłńóśźż]+$").tolist())        # przenośne: działa w każdym silniku

3.0.6 | str | ArrowStringArray
domyślnie             : [False, False, True]
flags=re.U (Python re): [True, True, True]
astype(object)        : [True, True, True]
jawna klasa liter PL  : [True, True, True]


---

## 8. Regex w Polars (inny silnik, ważne różnice)

Polars używa silnika **Rust `regex`**: bardzo szybki i bezpieczny (brak katastrofalnego cofania), ale **bez lookaround i bez odwołań wstecznych**. Podstawy (klasy, kwantyfikatory, grupy, kotwice, `(?i)`) działają jak w Pythonie.

| Cecha | Python `re` / pandas | Polars (Rust `regex`) |
|---|---|---|
| Lookahead / lookbehind `(?=…)` | ✅ | ❌ (błąd „look-around… not supported") |
| Odwołania wsteczne `\1` we wzorcu | ✅ | ❌ |
| Grupy nazwane | `(?P<n>…)` | `(?P<n>…)` i `(?<n>…)` |
| Zamiennik z grupą | `\1`, `\g<n>` | `$1`, `${n}` (dosłowny `$` zapisujesz `$$`) |
| Flagi | parametr `flags=` lub `(?i)` | tylko inline: `(?i)`, `(?m)`, `(?s)`, `(?x)` |
| Klasy Unicode `\p{L}` | ❌ w `re` | ✅ |
| `str.replace` | `regex=False` domyślnie | `literal=False` domyślnie (wzorzec jest regexem), `replace` zamienia **pierwsze** dopasowanie, `replace_all` wszystkie |

| Metoda Polars (`pl.col(...).str.`) | Parametry (✅ / ⬜) | Odpowiednik pandas |
|---|---|---|
| `contains(pattern, literal, strict)` | ✅ `pattern`; ⬜ `literal=False`, `strict=True` | `str.contains` |
| `extract(pattern, group_index)` | ✅ `pattern`; ⬜ `group_index=1` | `str.extract` (jedna grupa → Series) |
| `extract_groups(pattern)` | ✅ `pattern` (z grupami) | `str.extract` (wiele grup) → struktura; `.struct.unnest()` robi kolumny |
| `extract_all(pattern)` | ✅ `pattern` | `str.findall` (lista dopasowań) |
| `count_matches(pattern, literal)` | ✅ `pattern`; ⬜ `literal` | `str.count` |
| `replace(pattern, value, literal, n)` / `replace_all(...)` | ✅ `pattern`, ✅ `value`; ⬜ `literal`, `n` | `str.replace(regex=True)` |
| `find(pattern, literal)` | ✅ `pattern` | `str.find` (pozycja pierwszego dopasowania) |
| `starts_with`, `ends_with`, `strip_chars` | ✅ | `startswith`, `endswith`, `strip` |

In [19]:
pdf = pl.from_pandas(df)
pdf2 = pl.from_pandas(df2)

# filtr regexem i flagi inline
print(pdf.filter(pl.col("product_code").str.contains(r"^PRD-2026-"))["product_code"].to_list())
print(pdf.filter(pl.col("raw_name").str.contains(r"(?i)router|switch"))["raw_name"].to_list())

# extract_groups → struct → kolumny (odpowiednik pandas extract z nazwanymi grupami)
print(pdf.select(pl.col("product_code").str.extract_groups(r"(?P<typ>[A-Z]{3})-(?P<rok>\d{4})-(?P<nr>\d{5})").alias("p")).unnest("p"))

# extract (jedna grupa), extract_all, count_matches
print(pdf.select(region=pl.col("file_path").str.extract(r"region_(\w+)/", 1)))
print(pdf2.select(tagi=pl.col("comment").str.extract_all(r"#\w+"), ile=pl.col("comment").str.count_matches(r"#\w+")))

# replace_all z grupami ($1, ${nazwa}); zamiana formatu daty
print(pdf2.select(pl.col("date_text").str.replace(r"^(\d{1,2})\.(\d{1,2})\.(\d{4})$", "${3}-${2}-${1}")).to_series().to_list())

['PRD-2026-00123', 'PRD-2026-00456', 'PRD-2026-00234']
['  Router TP-Link AC1200 ', 'SWITCH netgear 8-port']
shape: (4, 3)
┌─────┬──────┬───────┐
│ typ ┆ rok  ┆ nr    │
│ --- ┆ ---  ┆ ---   │
│ str ┆ str  ┆ str   │
╞═════╪══════╪═══════╡
│ PRD ┆ 2026 ┆ 00123 │
│ PRD ┆ 2026 ┆ 00456 │
│ SVC ┆ 2025 ┆ 00789 │
│ PRD ┆ 2026 ┆ 00234 │
└─────┴──────┴───────┘
shape: (4, 1)
┌────────┐
│ region │
│ ---    │
│ str    │
╞════════╡
│ north  │
│ south  │
│ east   │
│ west   │
└────────┘
shape: (5, 2)
┌──────────────────────┬─────┐
│ tagi                 ┆ ile │
│ ---                  ┆ --- │
│ list[str]            ┆ u32 │
╞══════════════════════╪═════╡
│ ["#router", "#wifi"] ┆ 2   │
│ []                   ┆ 0   │
│ []                   ┆ 0   │
│ ["#promocja"]        ┆ 1   │
│ []                   ┆ 0   │
└──────────────────────┴─────┘
['2026-03-15', '2026-03-15', '15/03/2026', '2026-3-5', 'brak daty']


In [20]:
# co zrobić, gdy w Polars potrzebujesz lookaround (nie istnieje): przepisz wzorzec na grupę przechwytującą
print(pdf2.select(kwota=pl.col("comment").str.extract(r"(\d+) PLN", 1)))                   # zamiast (\d+)(?= PLN)

# lookahead w Polars → błąd (pokazane celowo)
try:
    pdf2.select(pl.col("comment").str.contains(r"\d+(?= PLN)"))
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0][:70])

# gdy naprawdę potrzebny lookaround/backreference: wykonaj ten krok w pandas/re (np. po to_pandas()) albo przez map_elements (wolne)
dup = pdf2.select(pl.col("comment").map_elements(lambda s: bool(re.search(r"\b(\w+)\s+\1\b", s)), return_dtype=pl.Boolean))
print(dup.to_series().to_list())

shape: (5, 1)
┌───────┐
│ kwota │
│ ---   │
│ str   │
╞═══════╡
│ null  │
│ null  │
│ null  │
│ 199   │
│ null  │
└───────┘
ComputeError - regex error: regex parse error:
[False, False, True, False, False]


---

## 9. Szablony wzorców (z testami) i przepisy do czyszczenia danych

### 9.1. Biblioteka wzorców walidacyjnych

Każdy szablon ma **przykłady pozytywne** (muszą pasować w całości) i **negatywne** (nie mogą). Komórka poniżej sprawdza je przez `re.fullmatch`; jeśli zmienisz wzorzec, od razu zobaczysz, co przestało działać. To też gotowy schemat testowania **Twoich** wzorców.

> Wzorce poniżej są **praktyczne**, a nie „idealne": np. e-mail według RFC jest absurdalnie skomplikowany; tu łapiemy 99% realnych adresów. Formaty dat/NIP-u sprawdzają **kształt**, a nie poprawność merytoryczną (np. 31 lutego, suma kontrolna NIP; sprawdzimy ją osobno w Pythonie).

In [21]:
TEMPLATES = {
    # nazwa: (wzorzec, [powinny pasować w całości], [nie powinny pasować])
    "email":             (r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", ["jan.kowalski@firma.pl", "a+b@x.io", "anna_nowak@firma.com.pl"], ["jan@", "@firma.pl", "jan firma.pl", "jan@firma"]),
    "telefon_pl":        (r"(?:\+48|0048)?[ -]?\d{3}[ -]?\d{3}[ -]?\d{3}", ["+48 601 234 567", "601-234-568", "601234567", "0048 601 234 567"], ["12-34", "601 234 56", "+49 601 234 567"]),
    "telefon_stacjonarny": (r"\(\d{2}\)\s?\d{3}[ -]?\d{2}[ -]?\d{2}", ["(22) 123 45 67", "(12)1234567"], ["22 123 45 67", "(2) 123 45 67"]),
    "kod_pocztowy_pl":   (r"\d{2}-\d{3}", ["00-950", "30-001"], ["80-1", "00950", "00-9500"]),
    "nip":               (r"\d{3}-?\d{3}-?\d{2}-?\d{2}", ["1234563218", "123-456-32-18"], ["123-45-678", "12345632"]),
    "iban_pl":           (r"PL\d{2}(?:\s?\d{4}){6}", ["PL61 1090 1014 0000 0712 1981 2874", "PL61109010140000071219812874"], ["PL61 1090", "DE61 1090 1014 0000 0712 1981 2874"]),
    "data_iso":          (r"\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])", ["2026-03-15", "1999-12-31"], ["2026-13-01", "2026-3-5", "2026-03-32"]),
    "data_pl":           (r"(?:0?[1-9]|[12]\d|3[01])[./](?:0?[1-9]|1[0-2])[./]\d{4}", ["15.03.2026", "5/3/2026", "31.12.1999"], ["32.01.2026", "15.13.2026", "15-03-2026"]),
    "godzina":           (r"(?:[01]\d|2[0-3]):[0-5]\d(?::[0-5]\d)?", ["08:15", "23:59:59", "00:00"], ["24:00", "8:15", "12:60"]),
    "liczba_calkowita":  (r"[+-]?\d+", ["7", "-42", "+2026"], ["1.5", "abc", ""]),
    "liczba_dziesietna": (r"[+-]?\d+(?:[.,]\d+)?", ["3.14", "-2,5", "42"], ["3.", ".5", "1,2,3"]),
    "kwota_pl":          (r"\d{1,3}(?:[ \xa0]\d{3})*(?:,\d{2})?", ["1 234,56", "12,50", "1 234 567,00", "12"], ["12 34,56", "1234 ,56", "12,5"]),
    "kwota_us":          (r"\d{1,3}(?:,\d{3})*(?:\.\d{2})?", ["1,234.56", "12.50", "1,234,567"], ["12,34.56", "1234,567"]),
    "procent":           (r"\d+(?:[.,]\d+)?\s?%", ["15%", "12,5 %", "0.5%"], ["%", "15"]),
    "url":               (r"https?://[^\s/$.?#][^\s]*", ["https://example.com", "http://sub.firma.pl/a/b?x=1#top"], ["example.com", "ftp://x.pl", "https://"]),
    "ipv4":              (r"(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)", ["192.168.0.1", "255.255.255.255", "0.0.0.0"], ["256.1.1.1", "1.2.3", "1.2.3.4.5"]),
    "uuid":              (r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}", ["123e4567-e89b-12d3-a456-426614174000"], ["123e4567-e89b-12d3-a456", "zzzzzzzz-e89b-12d3-a456-426614174000"]),
    "semver":            (r"\d+\.\d+\.\d+(?:-[\w.]+)?", ["1.2.3", "10.0.1-beta.2"], ["1.2", "v1.2.3"]),
    "kod_produktu":      (r"[A-Z]{3}-\d{4}-\d{5}", ["PRD-2026-00123", "SVC-2025-00789"], ["prd-2026-00123", "PRD-26-00123", "PRD-2026-123"]),
    "haslo_silne":       (r"(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}", ["Abcdefg1", "ZxcvbnM12345"], ["abcdefg1", "Abcdefgh", "Ab1"]),
}

bad = 0
for name, (pattern, good, nope) in TEMPLATES.items():
    comp = re.compile(pattern)
    fails = [g for g in good if not comp.fullmatch(g)] + [n for n in nope if comp.fullmatch(n)]
    bad += bool(fails)
    print(f"{'OK  ' if not fails else 'BŁĄD'} {name:22s} {pattern if len(pattern) < 62 else pattern[:59] + '…'}", "" if not fails else f"  -> {fails}")
print("\nszablonów:", len(TEMPLATES), "| z błędami:", bad)

OK   email                  [\w.+-]+@[\w-]+(?:\.[\w-]+)+ 
OK   telefon_pl             (?:\+48|0048)?[ -]?\d{3}[ -]?\d{3}[ -]?\d{3} 
OK   telefon_stacjonarny    \(\d{2}\)\s?\d{3}[ -]?\d{2}[ -]?\d{2} 
OK   kod_pocztowy_pl        \d{2}-\d{3} 
OK   nip                    \d{3}-?\d{3}-?\d{2}-?\d{2} 
OK   iban_pl                PL\d{2}(?:\s?\d{4}){6} 
OK   data_iso               \d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01]) 
OK   data_pl                (?:0?[1-9]|[12]\d|3[01])[./](?:0?[1-9]|1[0-2])[./]\d{4} 
OK   godzina                (?:[01]\d|2[0-3]):[0-5]\d(?::[0-5]\d)? 
OK   liczba_calkowita       [+-]?\d+ 
OK   liczba_dziesietna      [+-]?\d+(?:[.,]\d+)? 
OK   kwota_pl               \d{1,3}(?:[ \xa0]\d{3})*(?:,\d{2})? 
OK   kwota_us               \d{1,3}(?:,\d{3})*(?:\.\d{2})? 
OK   procent                \d+(?:[.,]\d+)?\s?% 
OK   url                    https?://[^\s/$.?#][^\s]* 
OK   ipv4                   (?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?… 
OK   uuid      

**NIP: format + suma kontrolna** (regex sprawdza kształt, arytmetyka poprawność): wagi `6 5 7 2 3 4 5 6 7`, suma ważona modulo 11 musi równać się ostatniej cyfrze.

In [22]:
nips = ["123-456-32-18", "1234563219", "526-104-08-28", "12-345"]
for n in nips:
    ok_format = re.fullmatch(r"\d{3}-?\d{3}-?\d{2}-?\d{2}", n) is not None
    digits = [int(c) for c in re.sub(r"\D", "", n)]
    ok_sum = ok_format and (sum(w * d for w, d in zip([6, 5, 7, 2, 3, 4, 5, 6, 7], digits[:9])) % 11 == digits[9])
    print(f"{n:15s} format: {ok_format!s:5s} suma kontrolna: {ok_sum}")

123-456-32-18   format: True  suma kontrolna: True
1234563219      format: True  suma kontrolna: False
526-104-08-28   format: True  suma kontrolna: True
12-345          format: False suma kontrolna: False


### 9.2. Przepisy: czyszczenie i wyciąganie danych (pandas)

| Zadanie | Technika |
|---|---|
| Ujednolicenie telefonów do `+48XXXXXXXXX` | usuń nie-cyfry (`\D`), potem `extract` ostatnich 9 cyfr |
| Kwota tekstowa → liczba (różne formaty) | `extract` fragmentu liczbowego + `np.select` po formacie, usunięcie separatorów, zamiana przecinka |
| Wiele formatów dat → jedna data | dwa `extract` (po formacie), `combine_first`, `pd.to_datetime` |
| Parsowanie logów | jedno `extract` z grupami nazwanymi → kolumny |
| Anonimizacja (maskowanie e-mail, telefon) | `re.sub`/`str.replace` z wzorcami szablonów |
| Lista słów kluczowych | `"\b(?:" + "\|".join(map(re.escape, słowa)) + ")\b"` |
| CamelCase ↔ snake_case | lookaround: `(?<!^)(?=[A-Z])` |
| Usuwanie polskich znaków | `str.normalize("NFKD")` + usunięcie znaków łączących `[\u0300-\u036f]` + mapa `ł→l` |
| Dzielenie CSV z cudzysłowami | `,(?=(?:[^"]*"[^"]*")*[^"]*$)` (przecinki poza cudzysłowami) |

In [23]:
# telefony → +48XXXXXXXXX
digits = df2["phone"].str.replace(r"\D", "", regex=True)                       # same cyfry
nine = digits.str.extract(r"(?:0048|48)?(\d{9})$", expand=False)               # ostatnie 9 cyfr, opcjonalny prefiks kraju
df2["phone_norm"] = "+48" + nine                                                # NaN + tekst = NaN (brak numeru zostaje brakiem)
print(df2[["phone", "phone_norm"]])

                 phone    phone_norm
0      +48 601 234 567  +48601234567
1          601-234-568  +48601234568
2       (22) 123 45 67  +48221234567
3  tel. 0048 601234569  +48601234569
4                 brak           NaN


In [24]:
# kwoty tekstowe (PL i US) → float
import numpy as np
num = df2["amount_text"].str.extract(r"(\d[\d\s.,]*\d|\d)", expand=False).str.replace(r"[\s\xa0]", "", regex=True)   # fragment liczbowy bez spacji
is_pl = num.str.contains(r",\d{2}$", na=False)                                  # przecinek jako część dziesiętna (1234,56 lub 1.234,56)
is_us = num.str.contains(r"\.\d{2}$", na=False)                                 # kropka jako część dziesiętna (1,234.56)
df2["amount"] = np.select([is_pl, is_us],
                          [num.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
                           num.str.replace(",", "", regex=False)], default=np.nan)
df2["amount"] = pd.to_numeric(df2["amount"], errors="coerce")
print(df2[["amount_text", "amount"]])

       amount_text      amount
0      1 234,56 zł     1234.56
1        12,50 PLN       12.50
2  1 234 567,00 zł  1234567.00
3        $1,234.56     1234.56
4        brak ceny         NaN


In [25]:
# wiele formatów dat → jedna kolumna datetime
iso = df2["date_text"].str.extract(r"^(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})$")
pl_ = df2["date_text"].str.extract(r"^(?P<day>\d{1,2})[./](?P<month>\d{1,2})[./](?P<year>\d{4})$")
parts = iso.combine_first(pl_).astype("Float64")                                # pierwszy niepusty z dwóch formatów
df2["date"] = pd.to_datetime(parts[["year", "month", "day"]], errors="coerce")
print(df2[["date_text", "date"]])

# parsowanie logów: jeden wzorzec z grupami nazwanymi → kolumny
logs = df2["log"].str.extract(r"^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (?P<level>[A-Z]+) (?P<msg>.*?) \(code=(?P<code>\d+)\)$")
logs["ts"] = pd.to_datetime(logs["ts"]); logs["code"] = pd.to_numeric(logs["code"])
print(logs)
print(logs.groupby("level").size())

    date_text       date
0  2026-03-15 2026-03-15
1  15.03.2026 2026-03-15
2  15/03/2026 2026-03-15
3    5.3.2026 2026-03-05
4   brak daty        NaT
                   ts  level                     msg   code
0 2026-03-15 10:21:05  ERROR  Brak połączenia z bazą  503.0
1 2026-03-15 10:21:09   INFO               Połączono  200.0
2 2026-03-15 10:22:41   WARN         Wolna odpowiedź  429.0
3 2026-03-15 10:25:00  ERROR                 Timeout  504.0
4                 NaT    NaN                     NaN    NaN
level
ERROR    2
INFO     1
WARN     1
dtype: int64


In [26]:
# anonimizacja: maskowanie e-maili i telefonów w komentarzach
masked = (df2["comment"]
          .str.replace(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", "[EMAIL]", regex=True)
          .str.replace(r"(?:\+48[ -]?)?\d{3}[ -]?\d{3}[ -]?\d{3}", "[TEL]", regex=True))
print(masked.tolist())

# lista słów kluczowych: re.escape + "|".join (słowa mogą zawierać znaki specjalne!)
KEYWORDS = ["router", "switch", "access point", "TP-Link"]
kw = r"\b(?:" + "|".join(map(re.escape, KEYWORDS)) + r")\b"
print(df["raw_name"].str.extract("(" + kw + ")", flags=re.I, expand=False).tolist())

# CamelCase ↔ snake_case, usuwanie polskich znaków
print(re.sub(r"(?<!^)(?=[A-Z])", "_", "UserAccountId").lower(), "|", re.sub(r"_(\w)", lambda m: m.group(1).upper(), "user_account_id"))
pl_text = pd.Series(["Zażółć gęślą jaźń", "ŁÓDŹ"])
print(pl_text.str.translate(str.maketrans("łŁ", "lL")).str.normalize("NFKD").str.replace("[\u0300-\u036f]", "", regex=True).tolist())   # zwykły (nie raw) napis: znaki wstawia Python, więc działa w każdym silniku

# CSV w jednym polu: dziel po przecinkach POZA cudzysłowami (lookahead)
print(re.split(r',(?=(?:[^"]*"[^"]*")*[^"]*$)', 'a,"b,c",d,"e"'))

['Świetny produkt!! polecam #router #wifi', 'Kontakt: [EMAIL] lub [TEL]', 'to to jest powtórzenie w tekście', 'Cena 199 PLN, rabat 15%, #promocja', 'Brak uwag']
['Router', 'SWITCH', nan, 'Access Point']
user_account_id | userAccountId
['Zazolc gesla jazn', 'LODZ']
['a', '"b,c"', 'd', '"e"']


---

## 10. Jak budować wzorzec krok po kroku

Sprawdzony proces (zamiast pisać długi wzorzec „na raz"):

| Krok | Co robisz |
|---|---|
| 1. **Próbki** | Zbierz 10–20 prawdziwych przykładów: te, które **mają** pasować, i te, które **nie mają** (najważniejsze są przypadki brzegowe i „śmieci") |
| 2. **Opis słowami** | „3 wielkie litery, myślnik, 4 cyfry, myślnik, 5 cyfr" |
| 3. **Wzorzec minimalny** | Zacznij od najprostszego, który łapie część przykładów |
| 4. **Ogranicz** | Dodawaj klasy, kwantyfikatory (`{n}`), kotwice (`^ $`, `\b`), aż negatywne przestaną pasować |
| 5. **Zgrupuj** | Dodaj grupy nazwane `(?P<...>)` dla fragmentów, które chcesz odczytać |
| 6. **Przetestuj** | Pętla po pozytywnych i negatywnych (jak w sekcji 9) |
| 7. **Udokumentuj** | Długie wzorce zapisz z `re.VERBOSE` (komentarze) |

In [27]:
good = ["PRD-2026-00123", "SVC-2025-00789"]                   # mają pasować
bad_ = ["prd-2026-00123", "PRD-26-00123", "PRD-2026-123", " PRD-2026-00123", "XYZ-2026-00123", "PRD-2026-001234"]   # nie mają pasować

steps = [
    r"PRD",                                                   # 1. za szeroko: pasuje do połowy śmieci
    r"[A-Z]{3}-\d{4}-\d{5}",                                  # 2. kształt, ale search znajdzie go także wewnątrz dłuższego tekstu
    r"^[A-Z]{3}-\d{4}-\d{5}$",                                # 3. kotwice: cały tekst
    r"^(?:PRD|SVC)-(?P<rok>20\d{2})-(?P<nr>\d{5})$",          # 4. tylko znane typy, rok 20xx, grupy nazwane
]
for p in steps:
    ok_good = sum(bool(re.search(p, g)) for g in good)         # search (nie fullmatch): pokazuje efekt kotwic
    bad_hit = [b for b in bad_ if re.search(p, b)]
    print(f"{p:50s} pozytywne: {ok_good}/{len(good)} | negatywne, które błędnie pasują: {bad_hit}")

PRD                                                pozytywne: 1/2 | negatywne, które błędnie pasują: ['PRD-26-00123', 'PRD-2026-123', ' PRD-2026-00123', 'PRD-2026-001234']
[A-Z]{3}-\d{4}-\d{5}                               pozytywne: 2/2 | negatywne, które błędnie pasują: [' PRD-2026-00123', 'XYZ-2026-00123', 'PRD-2026-001234']
^[A-Z]{3}-\d{4}-\d{5}$                             pozytywne: 2/2 | negatywne, które błędnie pasują: ['XYZ-2026-00123']
^(?:PRD|SVC)-(?P<rok>20\d{2})-(?P<nr>\d{5})$       pozytywne: 2/2 | negatywne, które błędnie pasują: []


### 10.1. `re.VERBOSE`: długi wzorzec z komentarzami

W trybie `re.X` spacje i komentarze `# …` we wzorcu są **ignorowane**, więc możesz go rozbić na linie. Dosłowną spację zapisz jako `[ ]` lub `\ `, a `#` jako `\#`.

### 10.2. Składanie wzorca z części

Powtarzające się fragmenty trzymaj w zmiennych i łącz konkatenacją lub f-stringiem (**uwaga:** w `rf"..."` klamry kwantyfikatorów `{4}` trzeba podwoić `{{4}}`, więc stałe fragmenty z kwantyfikatorami zapisuj w osobnych zmiennych).

In [28]:
phone = re.compile(r"""
    ^\s*
    (?P<prefix>(?:\+48|0048)[ -]?)?     # opcjonalny prefiks kraju (+48 lub 0048)
    (?P<a>\d{3})[ -]?                   # pierwsze 3 cyfry
    (?P<b>\d{3})[ -]?                   # drugie 3 cyfry
    (?P<c>\d{3})                        # ostatnie 3 cyfry
    \s*$
""", re.VERBOSE)
m = phone.fullmatch("+48 601 234 567")
print(m.groupdict())

# składanie z części
DD = r"(?:0[1-9]|[12]\d|3[01])"; MM = r"(?:0[1-9]|1[0-2])"; YYYY = r"\d{4}"
date_pl = rf"(?P<d>{DD})\.(?P<m>{MM})\.(?P<y>{YYYY})"
print(date_pl)
print(re.fullmatch(date_pl, "15.03.2026").groupdict(), re.fullmatch(date_pl, "32.03.2026"))

{'prefix': '+48 ', 'a': '601', 'b': '234', 'c': '567'}
(?P<d>(?:0[1-9]|[12]\d|3[01]))\.(?P<m>(?:0[1-9]|1[0-2]))\.(?P<y>\d{4})
{'d': '15', 'm': '03', 'y': '2026'} None


---

## 11. Pułapki i wydajność

| Pułapka | Objaw | Rozwiązanie |
|---|---|---|
| Brak `r"..."` | `"\d"` działa „przypadkiem", a `"\b"` to backspace | Zawsze surowe napisy |
| `.` nie pasuje do nowej linii | Wielolinijkowy tekst „urywa się" | `re.S` lub `[\s\S]` |
| `^`/`$` tylko na początku/końcu całego tekstu | Linie wewnątrz tekstu nie pasują | `re.M` |
| `$` pasuje **przed końcowym `\n`** | `abc\n` pasuje do `abc$` | `\Z` lub `fullmatch` |
| `match` zakotwicza tylko początek | „nie znajduje w środku" | `search` (gdziekolwiek) lub `fullmatch` (całość) |
| `\d` pasuje do cyfr z innych pism (Unicode) | `٣` liczone jako cyfra | `[0-9]` lub `re.A` |
| Zachłanność | `".*"` łapie od pierwszego do ostatniego | `.*?` lub `[^"]*` |
| `findall` z grupami | Zwraca krotki grup zamiast całych dopasowań | Grupy nieprzechwytujące `(?:…)` |
| Puste dopasowania | `\d*` dopasowuje „nic" | `\d+` (co najmniej jedna) |
| Znaki specjalne w danych użytkownika | `.`/`(` w szukanym tekście zmieniają wzorzec | `re.escape(tekst)` |
| Zamiennik z `\` | `re.sub(..., "\n")` wstawia nową linię | Surowy zamiennik `r"\1"` lub funkcja |
| pandas `str.replace` bez `regex=True` | Dosłowna zamiana | `regex=True` |
| Lookaround/backref w Polars | Błąd silnika | Zmień wzorzec lub zrób w pandas/`re` |
| **Katastrofalne cofanie** | Wzorzec „wisi" na niedopasowanym tekście | Poniżej |

In [29]:
# efekty pułapek
print(re.findall(r"\d*", "a1"))                                    # puste dopasowania: ['', '1', '']
print(re.search(r"abc$", "abc\n"), re.search(r"abc\Z", "abc\n"))   # $ pasuje przed końcowym \n, \Z nie
print(re.findall(r"\d", "٣4 5"), re.findall(r"[0-9]", "٣4 5"))      # \d łapie cyfrę z innego pisma
print(re.findall(r"(\d)(\d)", "1234"), re.findall(r"(?:\d)(?:\d)", "1234"))   # findall: krotki grup vs całe dopasowania

['', '1', '']
<re.Match object; span=(0, 3), match='abc'> None
['٣', '4', '5'] ['4', '5']
[('1', '2'), ('3', '4')] ['12', '34']


### 11.1. Katastrofalne cofanie (catastrophic backtracking)

Zagnieżdżone kwantyfikatory (`(a+)+`, `(.*)*`, `(\w+\s?)*$`) potrafią spowodować **wykładowy** wzrost liczby prób, gdy tekst **nie pasuje** na samym końcu. Czas rośnie ×4 co dwa dodatkowe znaki.

In [30]:
import time
for n in (16, 18, 20, 22):
    t = time.perf_counter(); re.match(r"(a+)+$", "a" * n + "!"); dt = time.perf_counter() - t       # zagnieżdżony kwantyfikator
    print(f"(a+)+$ dla n={n}: {dt:.3f} s")
t = time.perf_counter(); re.match(r"a+$", "a" * 100_000 + "!"); print(f"równoważny a+$ dla n=100000: {time.perf_counter() - t:.4f} s")

(a+)+$ dla n=16: 0.012 s
(a+)+$ dla n=18: 0.047 s
(a+)+$ dla n=20: 0.361 s
(a+)+$ dla n=22: 0.514 s
równoważny a+$ dla n=100000: 0.0014 s


**Jak się chronić:** nie zagnieżdżaj kwantyfikatorów na tych samych znakach; precyzuj klasy (`[^"]*` zamiast `.*`); używaj kwantyfikatorów zaborczych/grup atomowych (Python 3.11+: `a++`, `(?>…)`); ogranicz długość wejścia; dla niezaufanych danych używaj silnika bez cofania (**Polars/Rust `regex`**, Google RE2).

### 11.2. Wydajność w pandas i Polars

| Zasada | Dlaczego |
|---|---|
| Gdy wystarczy tekst dosłowny, użyj `regex=False` / `startswith` / `in` | Prostsze i bezpieczniejsze (bez znaków specjalnych); w typie `object` także szybsze, w pandas 3 + PyArrow różnica jest niewielka (zob. pomiar poniżej) |
| Używaj metod wektorowych `.str.*`, nie `apply(lambda …re.search…)` | Mniej narzutu Pythona: w pomiarze poniżej `apply` jest kilkanaście razy wolniejsze |
| Wiele wzorców naraz: jedna alternatywa `a\|b\|c` zamiast pętli po wzorcach | Jedno przejście po tekście |
| `re.compile` dla pętli własnych | Unika ponownej kompilacji |
| Polars dla dużych kolumn tekstowych | Silnik Rust, wielowątkowość; zysk rośnie z rozmiarem danych (przy 200 tys. wierszy wynik zbliżony do pandas + Arrow) |

In [31]:
import numpy as np
big = pd.Series(np.random.default_rng(0).choice(df["raw_name"].str.strip().to_numpy(), 200_000))
bigpl = pl.Series(big.to_numpy())

def timeit(label, f, repeat=3):
    best = min((lambda t0: (f(), time.perf_counter() - t0)[1])(time.perf_counter()) for _ in range(repeat))
    print(f"{label:48s} {best * 1000:8.1f} ms")

timeit("pandas contains regex=False", lambda: big.str.contains("router", case=False, regex=False))
timeit("pandas contains regex (alternatywa)", lambda: big.str.contains(r"router|switch|cable", case=False))
timeit("pandas apply(re.search)", lambda: big.apply(lambda s: bool(re.search(r"router|switch|cable", s, re.I))))
timeit("polars contains regex", lambda: bigpl.str.contains(r"(?i)router|switch|cable"))

pandas contains regex=False                          21.1 ms
pandas contains regex (alternatywa)                  21.6 ms
pandas apply(re.search)                             452.0 ms
polars contains regex                                18.4 ms


---

## 12. Debugowanie wzorców

| Narzędzie | Do czego |
|---|---|
| `re.sub(wzorzec, r"[[\g<0>]]", tekst)` | Zobacz, **co dokładnie** zostało dopasowane |
| `re.finditer(...)` + `m.span()`, `m.groupdict()` | Pozycje i grupy każdego dopasowania |
| `re.compile(p).groups`, `.groupindex` | Ile grup i jakie nazwy |
| `re.DEBUG` | Wypisuje drzewo składni wzorca (jak silnik go rozumie) |
| **regex101.com** (wybierz flavor **Python**) | Wyjaśnienie wzorca element po elemencie, podgląd grup, testy; **zastrzeżenie:** sprawdzaj flavor, bo Polars to Rust, a nie Python |
| Zmniejszaj wzorzec | Usuwaj elementy od końca, aż coś zacznie pasować; potem dodawaj po jednym |

In [32]:
p = re.compile(r"(?P<h>\d{2}):(?P<m>\d{2})", re.DEBUG)             # DEBUG wypisuje drzewo składni przy kompilacji
print(p.groups, p.groupindex)
print(re.sub(r"\d{2}:\d{2}", r"[[\g<0>]]", "od 08:15 do 17:45"))

SUBPATTERN 1 0 0
  MAX_REPEAT 2 2
    IN
      CATEGORY CATEGORY_DIGIT
LITERAL 58
SUBPATTERN 2 0 0
  MAX_REPEAT 2 2
    IN
      CATEGORY CATEGORY_DIGIT

 0. INFO 4 0b0 5 5 (to 5)
 5: MARK 0
 7. REPEAT_ONE 9 2 2 (to 17)
11.   IN 4 (to 16)
13.     CATEGORY UNI_DIGIT
15.     FAILURE
16:   SUCCESS
17: MARK 1
19. LITERAL 0x3a (':')
21. MARK 2
23. REPEAT_ONE 9 2 2 (to 33)
27.   IN 4 (to 32)
29.     CATEGORY UNI_DIGIT
31.     FAILURE
32:   SUCCESS
33: MARK 3
35. SUCCESS
2 {'h': 1, 'm': 2}
od [[08:15]] do [[17:45]]


---

## 13. Ćwiczenia z rozwiązaniami

Najpierw spróbuj sam (zadanie w tej komórce), potem sprawdź rozwiązanie w następnej. Rozwiązania mają `assert`, więc sprawdzą się same.

| # | Zadanie | Dane |
|---|---|---|
| 1 | Wyciągnij **rok** (liczba całkowita) z `product_code` | `df` |
| 2 | Które e-maile mają `_` lub `-` w części lokalnej (przed `@`)? | `df` |
| 3 | Zamień daty `DD.MM.RRRR` na `RRRR-MM-DD` (inne formaty zostaw bez zmian) | `df2["date_text"]` |
| 4 | Wyciągnij **numer wersji** z `file_path` (`_v2` → 2; brak → `NaN`) | `df` |
| 5 | Policz hashtagi w `comment` | `df2` |
| 6 | Wykryj komentarze z **powtórzonym słowem** pod rząd | `df2` |
| 7 | Wyciągnij kody błędów z linii logów o poziomie `ERROR` | `df2["log"]` |
| 8 | Wyciągnij poprawny kod pocztowy z `postal` (nieprawidłowe → `NaN`) | `df2` |

In [33]:
# rozwiązania
import warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")   # ćwiczenie 6 celowo używa grupy przechwytującej w contains

# 1
rok = df["product_code"].str.extract(r"-(\d{4})-", expand=False).astype(int)
assert rok.tolist() == [2026, 2026, 2025, 2026]

# 2
sep = df["email"].str.contains(r"^[^@]*[_-][^@]*@")
assert sep.tolist() == [False, True, False, True]

# 3
iso = df2["date_text"].str.replace(r"^(\d{1,2})\.(\d{1,2})\.(\d{4})$", lambda m: f"{m[3]}-{int(m[2]):02d}-{int(m[1]):02d}", regex=True)
assert iso.tolist()[:4] == ["2026-03-15", "2026-03-15", "15/03/2026", "2026-03-05"]

# 4
wersja = df["file_path"].str.extract(r"_v(\d+)\.", expand=False).astype(float)
assert wersja.isna().tolist() == [True, True, True, False] and wersja.iloc[3] == 2

# 5
tagi = df2["comment"].str.count(r"#\w+")
assert tagi.tolist() == [2, 0, 0, 1, 0]

# 6
powt = df2["comment"].str.contains(r"\b(\w+)\s+\1\b")
assert powt.tolist() == [False, False, True, False, False]

# 7
kody = df2["log"].str.extract(r"ERROR .*\(code=(\d+)\)", expand=False).dropna().astype(int)
assert kody.tolist() == [503, 504]

# 8
kod = df2["postal"].str.extract(r"\b(\d{2}-\d{3})\b", expand=False)
assert kod.tolist()[:2] == ["00-950", "30-001"] and pd.isna(kod[2]) and kod[3:].tolist() == ["50-123", "60-001"]

print("wszystkie ćwiczenia: OK")

wszystkie ćwiczenia: OK


---

## 14. Ściągawki

### 14.1. Składnia

| Element | Znaczenie |
|---|---|
| `.` | dowolny znak (bez `\n`) |
| `\d` `\D` | cyfra / nie-cyfra |
| `\w` `\W` | znak słowa (litera, cyfra, `_`) / odwrotność |
| `\s` `\S` | biały znak / odwrotność |
| `[abc]` `[a-z]` `[^abc]` | klasa znaków / zakres / negacja |
| `?` `*` `+` | 0–1 / 0+ / 1+ |
| `{n}` `{n,m}` `{n,}` | dokładnie n / od n do m / n lub więcej |
| `*?` `+?` `??` | wersje leniwe |
| `^` `$` | początek / koniec tekstu (linii z `re.M`) |
| `\A` `\Z` | ścisły początek / koniec tekstu |
| `\b` `\B` | granica wyrazu / nie-granica |
| `\|` | alternatywa |
| `( )` `(?: )` `(?P<n> )` | grupa przechwytująca / nieprzechwytująca / nazwana |
| `\1` `(?P=n)` | odwołanie wsteczne (tylko `re`/pandas) |
| `(?= )` `(?! )` `(?<= )` `(?<! )` | lookahead ± / lookbehind ± (tylko `re`/pandas) |
| `(?i)` `(?m)` `(?s)` `(?x)` | flagi inline |
| `\.` `\\` `\(` ... | dosłowne znaki specjalne |

### 14.2. Funkcje: `re` ↔ pandas ↔ Polars

| Zadanie | `re` | pandas | Polars |
|---|---|---|---|
| Czy występuje | `re.search` | `str.contains(pat, na=False)` | `str.contains(pat)` |
| Od początku | `re.match` | `str.match` | `str.contains(r"^…")` |
| Cały tekst | `re.fullmatch` | `str.fullmatch` | `str.contains(r"^…$")` |
| Pierwsze, grupy | `m.groups()` | `str.extract` | `str.extract` / `str.extract_groups` |
| Wszystkie | `re.findall` | `str.findall` / `str.extractall` | `str.extract_all` |
| Policz | `len(re.findall)` | `str.count` | `str.count_matches` |
| Zamień | `re.sub` | `str.replace(regex=True)` | `str.replace_all` (`$1`) |
| Podziel | `re.split` | `str.split(pat)` | `str.split(literal)` (regex: `extract_all`/`replace` + `split`) |

### 14.3. Silniki: co działa gdzie

| Cecha | Python `re` / pandas | Polars (Rust `regex`) | RE2 (Google) | PCRE (np. PHP, grep -P) |
|---|---|---|---|---|
| Lookaround | ✅ (lookbehind stałej długości) | ❌ | ❌ | ✅ |
| Odwołania wsteczne | ✅ | ❌ | ❌ | ✅ |
| Gwarancja braku katastrofalnego cofania | ❌ | ✅ | ✅ | ❌ |
| Grupy nazwane | `(?P<n>)` | `(?P<n>)` / `(?<n>)` | `(?P<n>)` | oba |
| Klasy `\p{L}` | ❌ (użyj `[^\W\d_]` lub modułu `regex`) | ✅ | ✅ | ✅ |
| `\w`, `\d`, `\b` domyślnie | Unicode | Unicode | **ASCII** (stąd różnice w pandas 3 + PyArrow, sekcja 7.2) | zależy od flag |

### 14.4. Którego narzędzia użyć

| Sytuacja | Wybór |
|---|---|
| Tekst dosłowny, bez wzorca | `str.contains(regex=False)`, `startswith`, `in` |
| Walidacja całego pola | `fullmatch` |
| Szukanie wzorca w tekście | `contains` / `search` |
| Rozbicie tekstu na kolumny | `str.extract` z grupami nazwanymi (pandas) / `extract_groups` (Polars) |
| Wiele wartości z jednego pola | `findall` / `extractall` / `extract_all` |
| Zamiana ze zmianą kolejności | `replace(regex=True)` z `\1` (pandas) / `$1` (Polars) lub funkcja zamiennika |
| Potrzebny lookaround/backreference na dużych danych | pandas lub `re`; w Polars przepisz wzorzec albo użyj `map_elements` |
| Duży zbiór, niezaufane dane | Polars (Rust, bez katastrofalnego cofania) |
| Przetwarzanie struktur (HTML, JSON, XML, pełny CSV, e-mail wg RFC) | **Nie regex**: parser (`BeautifulSoup`, `json`, `csv`, `email-validator`) |

### 14.5. Gdzie dalej

- Python HOWTO: *Regular Expression HOWTO* (docs.python.org) i dokumentacja modułu `re`.
- **regex101.com**: interaktywne testowanie z wyjaśnieniem (flavor Python).
- Pakiet **`regex`** (`pip install regex`): rozszerza `re` o `\p{L}`, lookbehind o zmiennej długości, dopasowania „rozmyte".
- Dokumentacja Polars: *Strings* / `str` namespace (składnia Rust `regex`).